# 1. Import Libraries

In [1]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# 2. Define Paths

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATA_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "data"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"
STREAMLIT_DIR = PROJECT_ROOT / "artifacts" / "streamlit"

STREAMLIT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed dir:", PROCESSED_DIR)
print("Data artifact dir:", DATA_ARTIFACT_DIR)
print("Streamlit dir:", STREAMLIT_DIR)

Project root: c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator
Processed dir: c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\data\processed
Data artifact dir: c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\artifacts\data
Streamlit dir: c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\artifacts\streamlit


In [3]:
TEST_PRICING_PATH = PROCESSED_DIR / "test_pricing.csv"

PORTFOLIO_KPI_PATH = DATA_ARTIFACT_DIR / "portfolio_kpi_summary.csv"
GRADE_PROFITABILITY_PATH = DATA_ARTIFACT_DIR / "grade_profitability_summary.csv"
POLICY_COMPARISON_PATH = DATA_ARTIFACT_DIR / "policy_comparison.csv"
SCENARIO_POLICY_PATH = DATA_ARTIFACT_DIR / "scenario_policy_summary.csv"
LGD_ASSUMPTIONS_PATH = DATA_ARTIFACT_DIR / "lgd_assumptions.csv"

source_paths = {
    "test_pricing": TEST_PRICING_PATH,
    "portfolio_kpi": PORTFOLIO_KPI_PATH,
    "grade_profitability": GRADE_PROFITABILITY_PATH,
    "policy_comparison": POLICY_COMPARISON_PATH,
    "scenario_policy_summary": SCENARIO_POLICY_PATH,
    "lgd_assumptions": LGD_ASSUMPTIONS_PATH
}

for name, path in source_paths.items():
    print(f"{name}: {path} | exists={path.exists()}")

test_pricing: c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\data\processed\test_pricing.csv | exists=True
portfolio_kpi: c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\artifacts\data\portfolio_kpi_summary.csv | exists=True
grade_profitability: c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\artifacts\data\grade_profitability_summary.csv | exists=True
policy_comparison: c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\artifacts\data\policy_comparison.csv | exists=True
scenario_policy_summary: c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\artifacts\data\scenario_policy_summary.csv | exists=True
lgd_assumptions: c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\artifacts\data\lgd_assumptions.csv | exists=True


# 3. Load Data

In [4]:
test_pricing = pd.read_csv(TEST_PRICING_PATH, low_memory=False, parse_dates=["issue_date"])

portfolio_kpi = pd.read_csv(PORTFOLIO_KPI_PATH)
grade_profitability = pd.read_csv(GRADE_PROFITABILITY_PATH)
policy_comparison = pd.read_csv(POLICY_COMPARISON_PATH)
scenario_policy_summary = pd.read_csv(SCENARIO_POLICY_PATH)
lgd_assumptions = pd.read_csv(LGD_ASSUMPTIONS_PATH)

print("test_pricing:", test_pricing.shape)
print("portfolio_kpi:", portfolio_kpi.shape)
print("grade_profitability:", grade_profitability.shape)
print("policy_comparison:", policy_comparison.shape)
print("scenario_policy_summary:", scenario_policy_summary.shape)
print("lgd_assumptions:", lgd_assumptions.shape)

display(test_pricing.head())

test_pricing: (59405, 62)
portfolio_kpi: (1, 24)
grade_profitability: (8, 29)
policy_comparison: (4, 21)
scenario_policy_summary: (16, 24)
lgd_assumptions: (8, 2)


,default_flag,loan_amnt,term_months,annual_inc_log,dti_capped,emp_length_num,emp_length_missing,open_acc,pub_rec,revol_bal_log,revol_util_capped,revol_util_missing,total_acc,mort_acc,mort_acc_missing,pub_rec_bankruptcies,pub_rec_bankruptcies_missing,credit_history_length_capped,home_ownership,verification_status,purpose,initial_list_status,application_type,loan_status,grade,sub_grade,int_rate,installment,issue_date,pd_raw,pd_sigmoid,calibrated_pd,internal_grade,actual_rate,ead,term_years,lgd,expected_loss,lifetime_expected_loss_rate,annualized_expected_loss_rate,required_rate,required_rate_capped,pricing_gap,pricing_status,interest_income,funding_cost,operating_cost,expected_profit,risk_adjusted_return,repriced_interest_income,repriced_expected_profit,repriced_risk_adjusted_return,capital_requirement,capital_charge,collection_cost,tail_risk_penalty,economic_profit,economic_return,repriced_economic_profit,repriced_economic_return,pricing_decision,strategy_approved
0,0,"5,000.0000",36.0000,10.0859,28.5500,4.0000,0,10.0000,1.0000,8.4065,49.0000,0,34.0000,4.0000,0,1.0000,0,24.5886,RENT,Not Verified,debt_consolidation,f,INDIVIDUAL,Fully Paid,C,C1,12.3900,167.0100,2015-02-01,0.2569,0.2372,0.2569,B,0.1239,"5,000.0000",3.0000,0.5000,642.2466,0.1284,0.0428,0.1528,0.1528,-0.0289,Underpriced,"1,858.5000",600.0000,300.0000,316.2534,0.0633,"2,292.2466",750.0000,0.1500,642.2466,154.1392,64.2247,57.7473,40.1423,0.0080,473.8889,0.0948,Approve at Current Rate,True
1,0,"25,000.0000",36.0000,11.3504,12.6500,7.0000,0,13.0000,0.0000,9.6637,43.5000,0,25.0000,2.0000,0,0.0000,0,10.9158,MORTGAGE,Verified,debt_consolidation,f,INDIVIDUAL,Fully Paid,D,D4,17.5700,898.4300,2014-08-01,0.1147,0.1176,0.1147,A,0.1757,"25,000.0000",3.0000,0.3500,"1,004.0153",0.0402,0.0134,0.1234,0.1234,0.0523,Overpriced,"13,177.5000","3,000.0000","1,500.0000","7,673.4847",0.3069,"9,254.0153","3,750.0000",0.1500,"1,004.0153",240.9637,100.4015,40.3219,"7,291.7977",0.2917,"3,368.3130",0.1347,Approve at Current Rate,True
2,0,"11,000.0000",36.0000,11.2898,17.6000,10.0000,0,19.0000,0.0000,10.2295,66.8000,0,44.0000,3.0000,0,0.0000,0,12.0000,MORTGAGE,Not Verified,debt_consolidation,w,INDIVIDUAL,Fully Paid,C,C1,14.3000,377.5600,2013-10-01,0.1286,0.1265,0.1286,A,0.1430,"11,000.0000",3.0000,0.3500,495.2668,0.0450,0.0150,0.1250,0.1250,0.0180,Overpriced,"4,719.0000","1,320.0000",660.0000,"2,243.7332",0.2040,"4,125.2668","1,650.0000",0.1500,495.2668,118.8640,49.5267,22.2990,"2,053.0435",0.1866,"1,459.3103",0.1327,Approve at Current Rate,True
3,1,"23,325.0000",60.0000,11.3386,31.8600,10.0000,0,16.0000,0.0000,6.8480,4.8000,0,26.0000,2.0000,0,0.0000,0,17.4155,MORTGAGE,Verified,debt_consolidation,f,INDIVIDUAL,Charged Off,B,B4,13.1100,532.0300,2013-04-01,0.3215,0.3136,0.3215,CCC,0.1311,"23,325.0000",5.0000,0.5500,"4,123.8173",0.1768,0.0354,0.1454,0.1454,-0.0143,Underpriced,"15,289.5375","4,665.0000","2,332.5000","4,168.2202",0.1787,"16,952.5673","5,831.2500",0.2500,"4,123.8173","1,649.5269",412.3817,463.9621,"1,642.3494",0.0704,"3,305.3792",0.1417,Approve at Current Rate,False
4,0,"10,000.0000",36.0000,11.2898,19.0800,1.0000,0,12.0000,0.0000,7.5507,8.0000,0,55.0000,3.0000,0,0.0000,0,12.0767,MORTGAGE,Not Verified,house,f,INDIVIDUAL,Fully Paid,A,A1,6.0300,304.3600,2013-03-01,0.0663,0.0908,0.0663,AAA,0.0603,"10,000.0000",3.0000,0.3000,198.9846,0.0199,0.0066,0.1166,0.1166,-0.0563,Underpriced,"1,809.0000","1,200.0000",600.0000,-189.9846,-0.0190,"3,498.9846","1,500.0000",0.1500,198.9846,47.7563,19.8985,4.6194,-262.2588,-0.0262,"1,427.7258",0.1428,Approve if Repriced,True


# 4. Streamlit Preparation

In [5]:
# Required streamlit columns

required_pricing_cols = [
    "loan_amnt",
    "term_months",
    "int_rate",
    "default_flag",
    "calibrated_pd",
    "internal_grade",
    "actual_rate",
    "required_rate",
    "pricing_gap",
    "pricing_status",
    "expected_loss",
    "expected_profit",
    "economic_profit",
    "economic_return",
    "pricing_decision",
    "strategy_approved"
]

missing_cols = [col for col in required_pricing_cols if col not in test_pricing.columns]

print("Missing required columns:", missing_cols)

if missing_cols:
    raise ValueError(f"Missing required Streamlit columns: {missing_cols}")

Missing required columns: []


In [6]:
# Lightweight pricing sample

SAMPLE_SIZE = 15000
RANDOM_STATE = 42

if len(test_pricing) <= SAMPLE_SIZE:
    pricing_sample = test_pricing.copy()
else:
    pricing_sample = (
        test_pricing
        .groupby("internal_grade", group_keys=False)
        .apply(
            lambda x: x.sample(
                n=max(1, int(SAMPLE_SIZE * len(x) / len(test_pricing))),
                random_state=RANDOM_STATE
            )
        )
        .reset_index(drop=True)
    )

print("Pricing sample shape:", pricing_sample.shape)

print("Grade distribution in full test:")
display(
    test_pricing["internal_grade"]
    .value_counts(normalize=True)
    .sort_index()
    .to_frame("full_share")
)

print("Grade distribution in sample:")
display(
    pricing_sample["internal_grade"]
    .value_counts(normalize=True)
    .sort_index()
    .to_frame("sample_share")
)

Pricing sample shape: (14998, 62)
Grade distribution in full test:


,full_share
internal_grade,
A,0.1939
AA,0.1429
AAA,0.0575
B,0.1176
BB,0.1301
BBB,0.2004
CCC,0.0898
D,0.0679


Grade distribution in sample:


,sample_share
internal_grade,
A,0.1939
AA,0.1429
AAA,0.0575
B,0.1176
BB,0.1301
BBB,0.2004
CCC,0.0897
D,0.0679


In [7]:
# Columns for Streamlit sample

streamlit_sample_cols = [
    # borrower / loan basics
    "loan_amnt",
    "term_months",
    "int_rate",
    "default_flag",
    "issue_date",

    # model outputs
    "calibrated_pd",
    "internal_grade",

    # pricing
    "actual_rate",
    "required_rate",
    "required_rate_capped",
    "pricing_gap",
    "pricing_status",
    "pricing_decision",

    # expected loss
    "ead",
    "lgd",
    "expected_loss",
    "lifetime_expected_loss_rate",
    "annualized_expected_loss_rate",

    # accounting profit
    "interest_income",
    "funding_cost",
    "operating_cost",
    "expected_profit",
    "risk_adjusted_return",

    # economic profit
    "capital_charge",
    "collection_cost",
    "tail_risk_penalty",
    "economic_profit",
    "economic_return",

    # repricing
    "repriced_expected_profit",
    "repriced_economic_profit",
    "repriced_economic_return",

    # strategy
    "strategy_approved",

    # benchmark
    "grade",
    "sub_grade"
]

streamlit_sample_cols = [col for col in streamlit_sample_cols if col in pricing_sample.columns]

streamlit_pricing_sample = pricing_sample[streamlit_sample_cols].copy()

print("Streamlit pricing sample shape:", streamlit_pricing_sample.shape)
display(streamlit_pricing_sample.head())

Streamlit pricing sample shape: (14998, 34)


,loan_amnt,term_months,int_rate,default_flag,issue_date,calibrated_pd,internal_grade,actual_rate,required_rate,required_rate_capped,pricing_gap,pricing_status,pricing_decision,ead,lgd,expected_loss,lifetime_expected_loss_rate,annualized_expected_loss_rate,interest_income,funding_cost,operating_cost,expected_profit,risk_adjusted_return,capital_charge,collection_cost,tail_risk_penalty,economic_profit,economic_return,repriced_expected_profit,repriced_economic_profit,repriced_economic_return,strategy_approved,grade,sub_grade
0,"12,000.0000",36.0000,14.4800,0,2015-12-01,0.1133,A,0.1448,0.1232,0.1232,0.0216,Overpriced,Approve at Current Rate,"12,000.0000",0.3500,475.8168,0.0397,0.0132,"5,212.8000","1,440.0000",720.0000,"2,576.9832",0.2147,114.1960,47.5817,18.8668,"2,396.3387",0.1997,"1,800.0000","1,619.3555",0.1349,True,C,C5
1,"6,000.0000",36.0000,11.9900,0,2011-06-01,0.1269,A,0.1199,0.1248,0.1248,-0.0049,Fairly Priced,Approve at Current Rate,"6,000.0000",0.3500,266.5395,0.0444,0.0148,"2,158.2000",720.0000,360.0000,811.6605,0.1353,63.9695,26.6540,11.8406,709.1965,0.1182,900.0000,797.5360,0.1329,True,B,B5
2,"2,250.0000",60.0000,13.4300,1,2011-02-01,0.1443,A,0.1343,0.1201,0.1201,0.0142,Overpriced,Approve at Current Rate,"2,250.0000",0.3500,113.6416,0.0505,0.0101,"1,510.8750",450.0000,225.0000,722.2334,0.3210,45.4567,11.3642,5.7397,659.6728,0.2932,562.5000,499.9394,0.2222,True,C,C3
3,"6,000.0000",36.0000,9.9100,0,2012-01-01,0.1186,A,0.0991,0.1238,0.1238,-0.0247,Underpriced,Approve at Current Rate,"6,000.0000",0.3500,249.0587,0.0415,0.0138,"1,783.8000",720.0000,360.0000,454.7413,0.0758,59.7741,24.9059,10.3384,359.7229,0.0600,900.0000,804.9817,0.1342,True,B,B1
4,"25,000.0000",36.0000,12.1800,0,2009-12-01,0.1134,A,0.1218,0.1232,0.1232,-0.0014,Fairly Priced,Approve at Current Rate,"25,000.0000",0.3500,992.1558,0.0397,0.0132,"9,135.0000","3,000.0000","1,500.0000","3,642.8442",0.1457,238.1174,99.2156,39.3749,"3,266.1364",0.1306,"3,750.0000","3,373.2921",0.1349,True,B,B4


In [8]:
# Dashboard KPI cards

def make_kpi_cards(portfolio_kpi_df):
    row = portfolio_kpi_df.iloc[0]

    kpi_cards = pd.DataFrame([
        {
            "metric": "Total Loans",
            "value": row["n_loans"],
            "format": "integer"
        },
        {
            "metric": "Total EAD",
            "value": row["total_ead"],
            "format": "currency"
        },
        {
            "metric": "Average PD",
            "value": row["avg_pd"],
            "format": "percent"
        },
        {
            "metric": "Actual Default Rate",
            "value": row["actual_default_rate"],
            "format": "percent"
        },
        {
            "metric": "Average Actual Rate",
            "value": row["avg_actual_rate"],
            "format": "percent"
        },
        {
            "metric": "Average Required Rate",
            "value": row["avg_required_rate"],
            "format": "percent"
        },
        {
            "metric": "Expected Loss",
            "value": row["total_expected_loss"],
            "format": "currency"
        },
        {
            "metric": "Expected Profit",
            "value": row["total_expected_profit"],
            "format": "currency"
        },
        {
            "metric": "Economic Profit",
            "value": row["total_economic_profit"],
            "format": "currency"
        },
        {
            "metric": "Economic Return",
            "value": row["portfolio_economic_return"],
            "format": "percent"
        },
        {
            "metric": "Underpriced Share",
            "value": row["underpriced_share"],
            "format": "percent"
        }
    ])

    return kpi_cards


streamlit_kpi_cards = make_kpi_cards(portfolio_kpi)

display(streamlit_kpi_cards)

,metric,value,format
0,Total Loans,"59,405.0000",integer
1,Total EAD,"838,615,300.0000",currency
2,Average PD,0.1960,percent
3,Actual Default Rate,0.1961,percent
4,Average Actual Rate,0.1366,percent
5,Average Required Rate,0.1351,percent
6,Expected Loss,"84,125,994.4379",currency
7,Expected Profit,"184,169,017.0996",currency
8,Economic Profit,"139,079,657.3065",currency
9,Economic Return,0.1658,percent


In [9]:
# Selected policy summary

SELECTED_POLICY = "Balanced"

selected_policy_summary = policy_comparison[
    policy_comparison["policy"].eq(SELECTED_POLICY)
].copy()

if selected_policy_summary.empty:
    raise ValueError(f"Selected policy '{SELECTED_POLICY}' not found in policy_comparison.")

display(selected_policy_summary)

,policy,max_pd,excluded_grades,approved_loans,rejected_loans,approval_rate,approved_ead,avg_pd,actual_default_rate,avg_actual_rate,avg_required_rate,total_expected_loss,total_expected_profit,portfolio_rar,total_capital_charge,total_collection_cost,total_tail_risk_penalty,total_economic_profit,portfolio_economic_return,underpriced_share,high_risk_share
1,Balanced,0.3000,D,49727,9678,0.8371,"667,123,675.0000",0.1571,0.1553,0.1282,0.1299,"44,890,819.3409","122,096,139.3141",0.1830,"12,993,471.6989","4,489,081.9341","3,095,361.2700","101,518,224.4112",0.1522,0.4344,0.1342


In [10]:
# Approved vs rejected summary from full test

In [11]:
def approval_segment_summary(df):
    summary = (
        df
        .assign(approval_segment=np.where(df["strategy_approved"], "Approved", "Rejected"))
        .groupby("approval_segment")
        .agg(
            n_loans=("default_flag", "size"),
            total_ead=("ead", "sum"),
            avg_pd=("calibrated_pd", "mean"),
            actual_default_rate=("default_flag", "mean"),
            avg_actual_rate=("actual_rate", "mean"),
            avg_required_rate=("required_rate", "mean"),
            total_expected_loss=("expected_loss", "sum"),
            total_expected_profit=("expected_profit", "sum"),
            total_economic_profit=("economic_profit", "sum"),
            avg_economic_return=("economic_return", "mean"),
            underpriced_share=("pricing_status", lambda x: (x == "Underpriced").mean()),
            high_risk_share=("internal_grade", lambda x: x.isin(["B", "CCC", "D"]).mean())
        )
        .reset_index()
    )

    return summary


streamlit_approval_summary = approval_segment_summary(test_pricing)

display(streamlit_approval_summary)

,approval_segment,n_loans,total_ead,avg_pd,actual_default_rate,avg_actual_rate,avg_required_rate,total_expected_loss,total_expected_profit,total_economic_profit,avg_economic_return,underpriced_share,high_risk_share
0,Approved,49727,"667,123,675.0000",0.1571,0.1553,0.1282,0.1299,"44,890,819.3409","122,096,139.3141","101,518,224.4112",0.1369,0.4344,0.1342
1,Rejected,9678,"171,491,625.0000",0.3961,0.4059,0.1798,0.1620,"39,235,175.0971","62,072,877.7854","37,561,432.8953",0.1840,0.2690,1.0000


In [12]:
# Compact grade summary

def approval_segment_summary(df):
    summary = (
        df
        .assign(approval_segment=np.where(df["strategy_approved"], "Approved", "Rejected"))
        .groupby("approval_segment")
        .agg(
            n_loans=("default_flag", "size"),
            total_ead=("ead", "sum"),
            avg_pd=("calibrated_pd", "mean"),
            actual_default_rate=("default_flag", "mean"),
            avg_actual_rate=("actual_rate", "mean"),
            avg_required_rate=("required_rate", "mean"),
            total_expected_loss=("expected_loss", "sum"),
            total_expected_profit=("expected_profit", "sum"),
            total_economic_profit=("economic_profit", "sum"),
            avg_economic_return=("economic_return", "mean"),
            underpriced_share=("pricing_status", lambda x: (x == "Underpriced").mean()),
            high_risk_share=("internal_grade", lambda x: x.isin(["B", "CCC", "D"]).mean())
        )
        .reset_index()
    )

    return summary


streamlit_approval_summary = approval_segment_summary(test_pricing)

display(streamlit_approval_summary)

,approval_segment,n_loans,total_ead,avg_pd,actual_default_rate,avg_actual_rate,avg_required_rate,total_expected_loss,total_expected_profit,total_economic_profit,avg_economic_return,underpriced_share,high_risk_share
0,Approved,49727,"667,123,675.0000",0.1571,0.1553,0.1282,0.1299,"44,890,819.3409","122,096,139.3141","101,518,224.4112",0.1369,0.4344,0.1342
1,Rejected,9678,"171,491,625.0000",0.3961,0.4059,0.1798,0.1620,"39,235,175.0971","62,072,877.7854","37,561,432.8953",0.1840,0.2690,1.0000


In [13]:
# Compact scenario summary

scenario_cols_for_streamlit = [
    "scenario",
    "policy",
    "funding_cost_rate",
    "operating_cost_rate",
    "target_margin_rate",
    "approval_rate",
    "approved_loans",
    "approved_ead",
    "avg_pd",
    "actual_default_rate",
    "avg_actual_rate",
    "avg_required_rate",
    "total_expected_loss",
    "total_expected_profit",
    "portfolio_rar",
    "total_economic_profit",
    "portfolio_economic_return",
    "total_repriced_economic_profit",
    "repriced_economic_return",
    "underpriced_share",
    "high_risk_share"
]

scenario_cols_for_streamlit = [
    col for col in scenario_cols_for_streamlit
    if col in scenario_policy_summary.columns
]

streamlit_scenario_policy_summary = scenario_policy_summary[scenario_cols_for_streamlit].copy()

display(streamlit_scenario_policy_summary.head())

,scenario,policy,funding_cost_rate,operating_cost_rate,target_margin_rate,approval_rate,approved_loans,approved_ead,avg_pd,actual_default_rate,avg_actual_rate,avg_required_rate,total_expected_loss,total_expected_profit,portfolio_rar,total_economic_profit,portfolio_economic_return,total_repriced_economic_profit,repriced_economic_return,underpriced_share,high_risk_share
0,Base,Conservative,0.0400,0.0200,0.0500,0.6125,36383,"472,913,075.0000",0.1256,0.1218,0.1198,0.1249,"22,171,888.7400","70,401,426.0250",0.1489,"61,283,265.3763",0.1296,"66,404,435.6012",0.1404,0.4659,0.0000
1,Base,Balanced,0.0400,0.0200,0.0500,0.8371,49727,"667,123,675.0000",0.1571,0.1553,0.1282,0.1299,"44,890,819.3409","122,096,139.3141",0.1830,"101,518,224.4112",0.1522,"93,121,861.3470",0.1396,0.4344,0.1342
2,Base,Growth,0.0400,0.0200,0.0500,0.9350,55545,"768,067,625.0000",0.1767,0.1760,0.1328,0.1326,"64,016,744.6039","157,799,454.6211",0.2054,"125,973,392.8634",0.1640,"104,937,379.4923",0.1366,0.4194,0.2249
3,Base,Aggressive,0.0400,0.0200,0.0500,0.9812,58289,"818,221,850.0000",0.1892,0.1888,0.1354,0.1343,"77,362,390.3878","176,585,978.4372",0.2158,"136,166,039.4935",0.1664,"108,502,951.0563",0.1326,0.4110,0.2614
4,Higher Funding Cost,Conservative,0.0600,0.0200,0.0500,0.6125,36383,"472,913,075.0000",0.1256,0.1218,0.1198,0.1449,"22,171,888.7400","40,192,387.5250",0.0850,"31,074,226.8763",0.0657,"66,404,435.6012",0.1404,0.6822,0.0000


In [14]:
# Methodology notes for Streamlit

methodology_notes = pd.DataFrame([
    {
        "section": "Final PD",
        "note": "Final PD is produced by raw XGBoost probability. Sigmoid calibration was tested but not selected because it worsened Brier Score and Expected Calibration Error."
    },
    {
        "section": "Internal Grade",
        "note": "Internal grades are derived from calibrated PD using validation-set supervised binning and validated on the holdout test set for monotonic default rates."
    },
    {
        "section": "Expected Loss",
        "note": "Expected Loss is calculated as PD × LGD × EAD. EAD is proxied by loan amount."
    },
    {
        "section": "LGD",
        "note": "LGD is assumption-based by internal grade because recovery and write-off recovery data are unavailable in the Lending Club dataset."
    },
    {
        "section": "Required Rate",
        "note": "Required Rate equals funding cost, operating cost, annualized expected loss rate, and target margin."
    },
    {
        "section": "Pricing Gap",
        "note": "Pricing Gap compares actual Lending Club interest rate against the required risk-based rate."
    },
    {
        "section": "Economic Profit",
        "note": "Economic Profit adjusts expected profit by subtracting capital charge, collection cost, and nonlinear tail-risk penalty."
    },
    {
        "section": "Policy Comparison",
        "note": "Approval policies are compared using approval rate, expected loss, economic profit, economic return, default rate, and high-risk exposure."
    },
    {
        "section": "Limitation",
        "note": "This is a portfolio analytics simulator, not a production regulatory credit decisioning system. LGD, capital cost, collection cost, and tail-risk penalty are simulation assumptions."
    }
])

display(methodology_notes)

,section,note
0,Final PD,Final PD is produced by raw XGBoost probabilit...
1,Internal Grade,Internal grades are derived from calibrated PD...
2,Expected Loss,Expected Loss is calculated as PD × LGD × EAD....
3,LGD,LGD is assumption-based by internal grade beca...
4,Required Rate,"Required Rate equals funding cost, operating c..."
5,Pricing Gap,Pricing Gap compares actual Lending Club inter...
6,Economic Profit,Economic Profit adjusts expected profit by sub...
7,Policy Comparison,Approval policies are compared using approval ...
8,Limitation,"This is a portfolio analytics simulator, not a..."


In [15]:
# Metadata JSON

metadata = {
    "project_name": "Risk-Based Loan Pricing & Profitability Simulator",
    "dataset": "Lending Club loan-level data",
    "modeling_unit": "loan-level observation",
    "final_pd_model": "XGBoost raw probability",
    "final_pd_column": "calibrated_pd",
    "internal_grade_method": "validation-set decision-tree supervised binning",
    "pricing_engine": {
        "expected_loss": "PD × LGD × EAD",
        "required_rate": "funding_cost + operating_cost + annualized_expected_loss_rate + target_margin",
        "pricing_gap": "actual_rate - required_rate",
        "economic_profit": "expected_profit - capital_charge - collection_cost - tail_risk_penalty"
    },
    "default_dashboard_policy": "Balanced",
    "streamlit_sample_size": int(len(streamlit_pricing_sample)),
    "full_test_rows": int(len(test_pricing)),
    "created_artifacts": [
        "streamlit_portfolio_kpi.csv",
        "streamlit_kpi_cards.csv",
        "streamlit_grade_profitability.csv",
        "streamlit_policy_comparison.csv",
        "streamlit_scenario_policy_summary.csv",
        "streamlit_approval_summary.csv",
        "streamlit_pricing_sample.csv",
        "streamlit_lgd_assumptions.csv",
        "streamlit_methodology_notes.csv",
        "streamlit_metadata.json"
    ]
}

print(json.dumps(metadata, indent=2))

{
  "project_name": "Risk-Based Loan Pricing & Profitability Simulator",
  "dataset": "Lending Club loan-level data",
  "modeling_unit": "loan-level observation",
  "final_pd_model": "XGBoost raw probability",
  "final_pd_column": "calibrated_pd",
  "internal_grade_method": "validation-set decision-tree supervised binning",
  "pricing_engine": {
    "expected_loss": "PD \u00d7 LGD \u00d7 EAD",
    "required_rate": "funding_cost + operating_cost + annualized_expected_loss_rate + target_margin",
    "pricing_gap": "actual_rate - required_rate",
    "economic_profit": "expected_profit - capital_charge - collection_cost - tail_risk_penalty"
  },
  "default_dashboard_policy": "Balanced",
  "streamlit_sample_size": 14998,
  "full_test_rows": 59405,
  "created_artifacts": [
    "streamlit_portfolio_kpi.csv",
    "streamlit_kpi_cards.csv",
    "streamlit_grade_profitability.csv",
    "streamlit_policy_comparison.csv",
    "streamlit_scenario_policy_summary.csv",
    "streamlit_approval_summary

# 5. Save Streamlit Artifacts

In [16]:
streamlit_portfolio_kpi_path = STREAMLIT_DIR / "streamlit_portfolio_kpi.csv"
streamlit_kpi_cards_path = STREAMLIT_DIR / "streamlit_kpi_cards.csv"
streamlit_grade_profitability_path = STREAMLIT_DIR / "streamlit_grade_profitability.csv"
streamlit_policy_comparison_path = STREAMLIT_DIR / "streamlit_policy_comparison.csv"
streamlit_scenario_policy_path = STREAMLIT_DIR / "streamlit_scenario_policy_summary.csv"
streamlit_approval_summary_path = STREAMLIT_DIR / "streamlit_approval_summary.csv"
streamlit_pricing_sample_path = STREAMLIT_DIR / "streamlit_pricing_sample.csv"
streamlit_lgd_path = STREAMLIT_DIR / "streamlit_lgd_assumptions.csv"
streamlit_methodology_path = STREAMLIT_DIR / "streamlit_methodology_notes.csv"
streamlit_metadata_path = STREAMLIT_DIR / "streamlit_metadata.json"

portfolio_kpi.to_csv(streamlit_portfolio_kpi_path, index=False)
streamlit_kpi_cards.to_csv(streamlit_kpi_cards_path, index=False)
streamlit_grade_profitability.to_csv(streamlit_grade_profitability_path, index=False)
policy_comparison.to_csv(streamlit_policy_comparison_path, index=False)
streamlit_scenario_policy_summary.to_csv(streamlit_scenario_policy_path, index=False)
streamlit_approval_summary.to_csv(streamlit_approval_summary_path, index=False)
streamlit_pricing_sample.to_csv(streamlit_pricing_sample_path, index=False)
lgd_assumptions.to_csv(streamlit_lgd_path, index=False)
methodology_notes.to_csv(streamlit_methodology_path, index=False)

with open(streamlit_metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved Streamlit artifacts:")
for path in [
    streamlit_portfolio_kpi_path,
    streamlit_kpi_cards_path,
    streamlit_grade_profitability_path,
    streamlit_policy_comparison_path,
    streamlit_scenario_policy_path,
    streamlit_approval_summary_path,
    streamlit_pricing_sample_path,
    streamlit_lgd_path,
    streamlit_methodology_path,
    streamlit_metadata_path
]:
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"{path.name}: {size_mb:.3f} MB")

NameError: name 'streamlit_grade_profitability' is not defined

In [ ]:
artifact_files = list(STREAMLIT_DIR.glob("*"))

artifact_summary = pd.DataFrame([
    {
        "file": path.name,
        "size_mb": path.stat().st_size / (1024 ** 2),
        "path": str(path)
    }
    for path in artifact_files
]).sort_values("file")

display(artifact_summary)

total_size_mb = artifact_summary["size_mb"].sum()

print(f"Total Streamlit artifact size: {total_size_mb:.3f} MB")

In [ ]:
# Final sample quality
print("Full test default rate:", test_pricing["default_flag"].mean())
print("Sample default rate:", streamlit_pricing_sample["default_flag"].mean())

print("\nFull grade distribution:")
display(
    test_pricing["internal_grade"]
    .value_counts(normalize=True)
    .sort_index()
    .to_frame("full_share")
)

print("\nSample grade distribution:")
display(
    streamlit_pricing_sample["internal_grade"]
    .value_counts(normalize=True)
    .sort_index()
    .to_frame("sample_share")
)

print("\nSample pricing status distribution:")
display(
    streamlit_pricing_sample["pricing_status"]
    .value_counts(normalize=True)
    .to_frame("sample_share")
)

print("\nSample strategy approved distribution:")
display(
    streamlit_pricing_sample["strategy_approved"]
    .value_counts(normalize=True)
    .rename(index={True: "Approved", False: "Rejected"})
    .to_frame("sample_share")
)